# Payment Health, Customer Pain & Recovery Dashboard 2025

Source dashboard link: https://datalens.ru/orn0iopf8jxs8

**Project idea:** CashFlow Shield - Payment Health, Customer Pain & Recovery.  
**Based on synthetic data.**  
This is an internal fintech initiative aimed at identifying and eliminating revenue losses caused by payment failures and repayment underperformance, while keeping risk, complaints, and operational losses under control.

In [1]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


## Case data

In [2]:
# Load libraries
import numpy as np
import pandas as pd
from pathlib import Path
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
# File paths
customers_1 = pd.read_csv('customers.csv')
customers_1.head(3)
# Customer

,customer_id,signup_date,customer_segment,risk_band,country_code,state_code,age_band,kyc_status,credit_score_band,acquisition_channel,consent_status,is_active
0,CUST_000001,2025-01-30,active,low,US,NY,35-44,verified,near_prime,referral,revoked,1
1,CUST_000002,2024-10-18,at_risk,low,US,GA,45-54,verified,prime,referral,active,1
2,CUST_000003,2024-10-20,high_value,medium,US,ENG,18-24,verified,subprime,paid_search,active,1


In [ ]:
# Data type check
customers_1.info()

# Min-max by column (3-year period), 2023-01-01 to 2025-12-30
customers_1['signup_date'].min(), customers_1['signup_date'].max()

In [4]:
# File paths
accounts_2 = pd.read_csv('accounts.csv')
accounts_2.head(3)
# Account

,account_id,customer_id,account_type,opened_date,account_status,currency,credit_limit_usd,autopay_enabled,statement_day,due_day
0,ACC_000001,CUST_004344,wallet,2024-08-03,open,USD,0,0,NaN,NaN
1,ACC_000002,CUST_003565,credit_card,2025-10-04,closed,USD,1000,1,23.0,28.0
2,ACC_000003,CUST_009431,wallet,2025-09-20,open,USD,0,1,NaN,NaN


In [ ]:
# Data type check
accounts_2.info()

# Min-max by column
accounts_2['opened_date'].min(), accounts_2['opened_date'].max()

In [5]:
# File paths
merchants_3 = pd.read_csv('merchants.csv')
merchants_3.head(3)
# Merchant / seller of goods or services (secondary table)

,merchant_id,merchant_name,merchant_category,merchant_country,payment_rail_preferred,risk_tier
0,MERCH_00001,Prime Pay 1,ecommerce,US,card,low
1,MERCH_00002,Pulse Services 2,travel,US,ach,medium
2,MERCH_00003,Atlas Market 3,telecom,CA,card,medium


In [ ]:
# Data type check
merchants_3.info()

In [6]:
# File paths
transactions_4 = pd.read_csv('transactions.csv')
transactions_4.head(3)
# A single financial operation: purchase, transfer, repayment, refund

,transaction_id,transaction_ts,customer_id,account_id,merchant_id,transaction_type,payment_rail,channel,device_type,amount_usd,...,geo_country,is_cross_border,merchant_category,status,decline_reason,fraud_flag,aml_flag,complaint_flag,collection_relevant_flag,experiment_group
0,TXN_0000001,2025-08-07 22:46:56,CUST_011942,ACC_004371,NaN,transfer,rtp,branch,desktop,43.92,...,US,0,p2p_transfer,refunded,merchant_refund,0,0,0,0,retry_email
1,TXN_0000002,2025-05-07 06:55:43,CUST_009712,ACC_007399,NaN,repayment,card,web,desktop,95.23,...,US,0,repayment,approved,NaN,0,0,0,0,control
2,TXN_0000003,2025-09-05 06:46:17,CUST_000918,ACC_000597,NaN,repayment,rtp,web,ios,91.45,...,US,0,repayment,approved,NaN,0,0,0,0,control


In [ ]:
# Data type check
transactions_4.info()

# Min-max by column (1-year period), 2025-01-01 to 2025-12-30
transactions_4['transaction_ts'].min(), transactions_4['transaction_ts'].max()

In [8]:
# File paths
complaints_5 = pd.read_csv('complaints.csv')
complaints_5.head(3)
# Customer complaint.
# Real user pain: which operation types lead to complaints more often,
# and where the problem is in the process or UX.

,complaint_id,complaint_created_date,customer_id,transaction_id,product,issue,sub_issue,complaint_channel,narrative_category,company_response,timely_response,dispute_flag,resolution_days,severity
0,CMP_000001,2025-10-07,CUST_003996,TXN_0000243,payments,dispute,unexpected_reversal,app,fee_dispute,closed_with_non_monetary_relief,0,0,16,low
1,CMP_000002,2025-03-28,CUST_000910,TXN_0000283,credit_card,payment_problem,card_or_account_declined,phone,failed_payment,closed_with_explanation,0,0,16,medium
2,CMP_000003,2025-03-22,CUST_008445,TXN_0000284,money_transfer,payment_problem,card_or_account_declined,phone,failed_payment,closed_with_explanation,1,0,34,low


In [ ]:
# Data type check
complaints_5.info()

# Min-max by column (1-year period), 2025-01-02 to 2026-01-19
complaints_5['complaint_created_date'].min(), complaints_5['complaint_created_date'].max()

In [9]:
# File paths
collections_cases_6 = pd.read_csv('collections_cases.csv')
collections_cases_6.head(3)
# Collection / reminder case.
# Used for repayment performance analysis, reminder channels,
# and operational efficiency.

,case_id,customer_id,account_id,related_transaction_id,case_open_date,due_date,outstanding_amount_usd,dpd_bucket,collection_stage,reminder_channel,promise_to_pay_flag,resolved_flag,resolution_date
0,COL_000001,CUST_005016,ACC_012132,TXN_0012258,2025-06-10,2025-06-27,145.58,8-30,late_collections,call,0,0,NaN
1,COL_000002,CUST_011132,ACC_009219,NaN,2025-04-03,2025-04-20,161.77,31-60,soft_reminder,push,0,0,NaN
2,COL_000003,CUST_009037,ACC_010056,NaN,2025-06-06,2025-06-22,81.94,1-7,soft_reminder,sms,0,1,2025-06-10


In [ ]:
# Data type check
collections_cases_6.info()

# Min-max by column
collections_cases_6['case_open_date'].min(), collections_cases_6['case_open_date'].max()

In [10]:
# File paths
risk_alerts_7 = pd.read_csv('risk_alerts.csv')
risk_alerts_7.head(3)
# Red flag for manual review:
# false positives, load on operations, review time,
# and growth-risk trade-off (secondary table).

,alert_id,transaction_id,customer_id,alert_ts,alert_type,risk_score,alert_status,investigation_outcome,review_time_minutes
0,RAL_000001,TXN_0001647,CUST_005490,2025-01-24 06:16:15,high_amount_review,81,new,pending,350
1,RAL_000002,TXN_0027616,CUST_007794,2025-08-14 15:27:36,manual_review,55,reviewed,operations_issue,191
2,RAL_000003,TXN_0099855,CUST_008497,2025-10-15 10:47:58,high_amount_review,87,new,false_positive,421


In [ ]:
# Data type check
risk_alerts_7.info()

# Min-max by column
risk_alerts_7['alert_ts'].min(), risk_alerts_7['alert_ts'].max()

In [11]:
# File paths
experiments_8 = pd.read_csv('experiments.csv')
experiments_8.head(3)
# Experiments (secondary table)

,assignment_id,customer_id,experiment_name,variant,assigned_date,eligible_flag,exposure_flag,primary_metric,converted_flag
0,EXP_000001,CUST_005316,payment_retry_nudge,control,2025-12-07,1,0,retry_success_rate,1
1,EXP_000002,CUST_001816,repayment_reminder_timing,same_day_push,2025-11-05,1,1,on_time_repayment_rate,1
2,EXP_000003,CUST_010469,repayment_reminder_timing,t_minus_3_email,2025-12-01,1,1,on_time_repayment_rate,0


In [ ]:
# Data type check
experiments_8.info()

# Min-max by column
experiments_8['assigned_date'].min(), experiments_8['assigned_date'].max()

In [12]:
# File paths
consent_events_9 = pd.read_csv('consent_events.csv')
consent_events_9.head(3)
# Consents (secondary table)

,consent_event_id,customer_id,consent_ts,consent_action,scope,channel,provider_name
0,CONS_000001,CUST_006523,2025-09-24 13:19:11,granted,balances,mobile_app,fdx_aggregator_b
1,CONS_000002,CUST_004689,2025-10-06 23:55:09,granted,payments,web,fdx_aggregator_a
2,CONS_000003,CUST_004796,2025-05-27 23:06:21,expired,transactions,mobile_app,fdx_aggregator_a


In [ ]:
# Data type check
consent_events_9.info()

# Min-max by column
consent_events_9['consent_ts'].min(), consent_events_9['consent_ts'].max()

## 6 marts - source layer for the dashboard + stakeholder use cases

### mart_executive_kpis_daily
1. As an executive, I want to see daily GPV, payment success, complaint pressure, and recovery efficiency in one place, so that I can assess overall business health in seconds.
2. As a business leader, I want to track KPI trends over time, so that I can quickly detect negative shifts before they become major losses.
3. As a decision-maker, I want to compare payment health with complaint and recovery signals, so that I can balance growth, service quality and collections performance.
4. As a CEO or GM, I want to use one executive dashboard instead of multiple reports, so that I can reduce decision latency and focus on priority actions.
5. As a leadership team member, I want to identify which KPI is underperforming today, so that I can immediately route the issue to the right team.

### mart_payment_funnel_daily
1. As a Head of Payments, I want to see how many payment attempts convert into approved transactions, so that I can monitor cash-in efficiency.
2. As a product manager, I want to monitor lost payment value by channel and payment rail, so that I can prioritize the most painful funnel leaks.
3. As a BI analyst, I want to track payment success by segment and transaction type, so that I can identify underperforming user journeys.
4. As an operations lead, I want to see whether payment performance is deteriorating on specific days, so that I can escalate technical or provider-related issues quickly.
5. As a growth analyst, I want to quantify how much revenue is lost due to failed or declined payments, so that I can support prioritization with financial impact.

### mart_decline_reason_daily
1. As a payments manager, I want to rank decline reasons by lost value, so that I can focus on the issues that cost the business the most money.
2. As a system analyst, I want to compare decline reasons across payment rails and device types, so that I can isolate where the process is breaking.
3. As a product owner, I want to identify whether the main issue is user behavior, technical failure, or risk control, so that I can assign the right fix owner.
4. As a support lead, I want to anticipate which payment issues may generate customer complaints, so that I can prepare response scripts and escalation rules.
5. As an engineering manager, I want to detect recurring decline clusters by platform context, so that I can investigate systemic defects faster.

### mart_complaints_voc_daily
1. As a customer experience lead, I want to see which complaint issues occur most often relative to transaction volume, so that I can focus on the most systemic customer pain points.
2. As a compliance or support manager, I want to track complaint severity and resolution speed, so that I can ensure service quality does not deteriorate.
3. As a product analyst, I want to connect complaint trends with payment issues, so that I can show that customer pain is linked to measurable process failures.
4. As a business stakeholder, I want to monitor complaint pressure over time, so that I can see whether recent fixes actually improved customer outcomes.
5. As a support operations lead, I want to identify high-severity complaint topics with long resolution times, so that I can rebalance service capacity and escalation rules.

### mart_complaints_by_payment_context
1. As a product manager, I want to know which payment rails and channels generate the most complaints, so that I can fix the most painful journey first.
2. As a payments team lead, I want to compare complaint volumes across merchant categories, so that I can identify where the payment experience is weakest.
3. As a journey owner, I want to map complaints to payment context, so that I can move from generic complaint reporting to concrete process fixes.
4. As a BI designer, I want to show complaint pressure as a context map rather than a flat list, so that decision-makers can quickly spot where the experience breaks.
5. As an executive, I want to see which combinations of rail, channel and merchant type create the most friction, so that I can prioritize cross-functional fixes.

### mart_repayment_funnel_daily
1. As a collections lead, I want to see how many cases move from open to resolved, so that I can assess recovery performance.
2. As a finance stakeholder, I want to track recovered value by DPD bucket, so that I can understand where collections efforts deliver the most money back.
3. As a strategy owner, I want to compare cure rate across segments and account types, so that I can refine recovery policies by customer profile.
4. As an operations manager, I want to monitor whether recovery performance is weakening over time, so that I can adjust contact and escalation tactics early.
5. As an executive, I want to know how much at-risk value the business is able to recover, so that I can judge whether the recovery engine offsets payment leakage.

## Payment funnel mart_1 - `build_mart_payment_funnel_daily`

Shows where cash-in is lost by rail, channel, operation type, and customer segment; where to improve reliability, where to change UX, and where risk gating may be too aggressive.

In [13]:
def build_mart_payment_funnel_daily(conn, out_dir="marts"):
    query = """
    SELECT
        c.customer_id,
        DATE_TRUNC('day', CAST(t.transaction_ts AS TIMESTAMP)) AS dt,
        c.customer_segment,
        t.payment_rail,
        t.channel,
        t.transaction_type,
        COUNT(*) AS attempts,
        SUM(CASE WHEN t.status = 'approved' THEN 1 ELSE 0 END) AS approved_cnt,
        SUM(CASE WHEN t.status = 'failed' THEN 1 ELSE 0 END) AS failed_cnt,
        SUM(CASE WHEN t.status = 'declined' THEN 1 ELSE 0 END) AS declined_cnt,
        ROUND(
            1.0 * SUM(CASE WHEN t.status = 'approved' THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0),
            4
        ) AS success_rate,
        SUM(CASE WHEN t.status IN ('failed', 'declined') THEN t.amount_usd ELSE 0 END) AS lost_value_usd,
        SUM(CASE WHEN t.status = 'approved' THEN t.fee_usd ELSE 0 END) AS fee_revenue_usd
    FROM transactions_4 t
    LEFT JOIN customers_1 c
      ON t.customer_id = c.customer_id
    GROUP BY 1, 2, 3, 4, 5, 6
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    try:
        df = conn.execute(query).df()
    except Exception as e:
        raise RuntimeError(f"SQL execution error: {e}") from e

    expected_columns = [
        "dt",
        "customer_segment",
        "payment_rail",
        "channel",
        "transaction_type",
        "attempts",
        "approved_cnt",
        "failed_cnt",
        "declined_cnt",
        "success_rate",
        "lost_value_usd",
        "fee_revenue_usd",
        "customer_id",
    ]
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in the resulting DataFrame: {missing_columns}")

    output_file = out_path / "mart_payment_funnel_daily.parquet"
    try:
        df.to_parquet(output_file, index=False)
    except Exception as e:
        raise RuntimeError(f"Parquet save error: {e}") from e

    return df, str(output_file)

conn = duckdb.connect()
conn.register("transactions", transactions_4)
conn.register("customers", customers_1)
df_payment_funnel, output_path = build_mart_payment_funnel_daily(conn)
display(df_payment_funnel.head(3))

,customer_id,dt,customer_segment,payment_rail,channel,transaction_type,attempts,approved_cnt,failed_cnt,declined_cnt,success_rate,lost_value_usd,fee_revenue_usd
0,CUST_011942,2025-08-07,active,rtp,branch,transfer,1,0.0,0.0,0.0,0.0,0.0,0.00
1,CUST_009712,2025-05-07,active,card,web,repayment,1,1.0,0.0,0.0,1.0,0.0,1.71
2,CUST_000918,2025-09-05,delinquent,rtp,web,repayment,1,1.0,0.0,0.0,1.0,0.0,0.50


In [14]:
#5
# Pivot table for lost_value_usd by dates {for Dashboard}
Lost_Value = pd.pivot_table(
    df_payment_funnel,
    index=['dt', 'customer_id'],
    values=['lost_value_usd'],
    aggfunc='sum',
    margins=True
).sort_values(by='lost_value_usd', ascending=False).head(3)
Lost_Value

# Lost_Value — monetary value lost due to unsuccessful payments
# [transactions.status in ('failed','declined')]

,,lost_value_usd
dt,customer_id,
All,,1264290.44
2025-09-12 00:00:00,CUST_000953,2088.33
2025-08-12 00:00:00,CUST_001655,2035.14


## Reasons for failed/declined transactions mart_2 -`build_mart_decline_reason_daily`

Shows which decline reasons drive the highest lost payment value.

In [15]:
def build_mart_decline_reason_daily(conn, out_dir="marts"):
    query = """
    SELECT
        c.customer_id,
        DATE_TRUNC('day', CAST(t.transaction_ts AS TIMESTAMP)) AS dt,
        COALESCE(NULLIF(t.decline_reason, ''), 'no_reason') AS decline_reason,
        t.payment_rail,
        t.device_type,
        c.customer_segment,
        COUNT(*) AS failed_count,
        SUM(t.amount_usd) AS failed_value_usd,
        ROUND(
            1.0 * COUNT(*) /
            NULLIF(
                SUM(COUNT(*)) OVER (
                    PARTITION BY DATE_TRUNC('day', CAST(t.transaction_ts AS TIMESTAMP))
                ), 0
            ),
            4
        ) AS share_of_failures
    FROM transactions_4 t
    LEFT JOIN customers_1 c
      ON t.customer_id = c.customer_id
    WHERE t.status IN ('failed','declined')
    GROUP BY 1, 2, 3, 4, 5, 6
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    try:
        df = conn.execute(query).df()
    except Exception as e:
        raise RuntimeError(f"SQL execution error: {e}") from e

    expected_columns = [
        "dt",
        "decline_reason",
        "payment_rail",
        "device_type",
        "customer_segment",
        "failed_count",
        "failed_value_usd",
        "share_of_failures",
        "customer_id",
    ]
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in the resulting DataFrame: {missing_columns}")

    output_file = out_path / "mart_decline_reason_daily.parquet"
    try:
        df.to_parquet(output_file, index=False)
    except Exception as e:
        raise RuntimeError(f"Parquet save error: {e}") from e

    return df, str(output_file)

conn = duckdb.connect()
conn.register("transactions", transactions_4)
conn.register("customers", customers_1)
df_decline_reason, output_path = build_mart_decline_reason_daily(conn)
display(df_decline_reason.head(3))

,customer_id,dt,decline_reason,payment_rail,device_type,customer_segment,failed_count,failed_value_usd,share_of_failures
0,CUST_007765,2025-01-03,insufficient_funds,card,desktop,delinquent,1,22.43,0.0357
1,CUST_004668,2025-01-03,processor_down,ach,ios,delinquent,1,222.78,0.0357
2,CUST_004673,2025-01-03,duplicate_attempt,card,ios,high_value,1,56.67,0.0357


In [17]:
#6
# Pivot table for Top_Decline_Reason by reasons and dates {for Dashboard}
Top_Decline_Reason = pd.pivot_table(
    df_decline_reason,
    index=['decline_reason', 'dt', 'customer_id'],
    values=['failed_value_usd'],
    aggfunc='sum',
    margins=True
).sort_values(by='failed_value_usd', ascending=False).head(3)
Top_Decline_Reason

# Top_Decline_Reason — reasons for payment value loss
# [transactions.status in ('failed','declined') & MAX(failed_value_usd)]

,,,failed_value_usd
decline_reason,dt,customer_id,
All,NaT,,1264290.44
duplicate_attempt,2025-09-12,CUST_000953,2088.33
processor_down,2025-08-12,CUST_001655,2035.14


## Voice-of-customer mart_3 - `build_mart_complaints_voc_daily`

Which products create reputational risk and cost-to-serve, and how that affects response speed and issue resolution time.

In [18]:
def build_mart_complaints_voc_daily(conn, out_dir="marts"):
    query = """
    WITH tx_day AS (
        SELECT
            DATE_TRUNC('day', CAST(transaction_ts AS TIMESTAMP)) AS dt,
            COUNT(*) AS tx_cnt_day
        FROM transactions
        GROUP BY 1
    ),
    cmp_issue_day AS (
        SELECT
            DATE_TRUNC('day', CAST(complaint_created_date AS TIMESTAMP)) AS dt,
            issue,
            COUNT(*) AS complaints_cnt,
            COUNT(DISTINCT customer_id) AS affected_customers_cnt,
            MIN(customer_id) AS sample_customer_id
        FROM complaints
        GROUP BY 1, 2
    ),
    ranked AS (
        SELECT
            c.dt,
            c.issue,
            c.complaints_cnt,
            t.tx_cnt_day,
            ROUND(c.complaints_cnt * 1000.0 / NULLIF(t.tx_cnt_day, 0), 2) AS complaint_rate_top_issue_per_1k_tx,
            c.affected_customers_cnt,
            c.sample_customer_id,
            ROW_NUMBER() OVER (
                PARTITION BY c.dt
                ORDER BY
                    c.complaints_cnt * 1000.0 / NULLIF(t.tx_cnt_day, 0) DESC,
                    c.complaints_cnt DESC,
                    c.issue
            ) AS rn
        FROM cmp_issue_day c
        LEFT JOIN tx_day t
          ON c.dt = t.dt
    )
    SELECT
        dt,
        issue,
        complaints_cnt,
        tx_cnt_day,
        complaint_rate_top_issue_per_1k_tx,
        affected_customers_cnt,
        sample_customer_id AS customer_id
    FROM ranked
    WHERE rn = 1
    ORDER BY dt
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    try:
        df = conn.execute(query).df()
    except Exception as e:
        raise RuntimeError(f"SQL execution error: {e}") from e

    expected_columns = [
        "dt",
        "issue",
        "complaints_cnt",
        "tx_cnt_day",
        "complaint_rate_top_issue_per_1k_tx",
        "affected_customers_cnt",
        "customer_id",
    ]
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in the resulting DataFrame: {missing_columns}")

    output_file = out_path / "mart_complaints_voc_daily.parquet"
    try:
        df.to_parquet(output_file, index=False)
    except Exception as e:
        raise RuntimeError(f"Parquet save error: {e}") from e

    return df, str(output_file)

conn = duckdb.connect()
conn.register("complaints", complaints_5)
conn.register("transactions", transactions_4)
df_complaints_voc, output_path = build_mart_complaints_voc_daily(conn)
display(df_complaints_voc.head(3))

,dt,issue,complaints_cnt,tx_cnt_day,complaint_rate_top_issue_per_1k_tx,affected_customers_cnt,customer_id
0,2025-01-02,payment_problem,1,248,4.03,1,CUST_000271
1,2025-01-06,service,1,256,3.91,1,CUST_004310
2,2025-01-07,service,1,310,3.23,1,CUST_004019


In [ ]:
# VOC logic:
# 1. Count all transactions by day.
# 2. Count complaints by day and issue.
# 3. For each issue, calculate COUNT(complaints) / COUNT(transactions) * 1000.
# 4. Within each day, keep the issue with the maximum rate.

# KPI logic (used later in mart 5):
# all complaints for the day / all transactions for the day * 1000

In [19]:
#7
# Pivot table for Top_Complaint_Driver by issue and date
Top_Complaint_Driver = pd.pivot_table(
    df_complaints_voc,
    index=['issue', 'dt', 'customer_id'],
    values=['complaint_rate_top_issue_per_1k_tx'],
    aggfunc='mean',
    margins=True
).sort_values(by='complaint_rate_top_issue_per_1k_tx', ascending=False).head(3)
Top_Complaint_Driver

# Top_Complaint_Driver — which issue hurts customer experience the most

complaint_rate_top_issue_per_1k_tx
issue           dt         customer_id                                    
payment_problem 2025-04-12 CUST_001871                               46.88
                2025-02-24 CUST_001200                               37.45
                2025-08-09 CUST_001264                               36.90

## Complaints / payment context mart_4 - `build_mart_complaints_by_payment_context`

Shows not abstract customer pain, but the exact process that creates it.

In [20]:
def build_mart_complaints_by_payment_context(conn, out_dir="marts"):
    query = """
    SELECT
        c.customer_id,
        DATE_TRUNC('day', CAST(cpl.complaint_created_date AS TIMESTAMP)) AS dt,
        cpl.issue,
        t.payment_rail,
        t.channel,
        t.merchant_category,
        c.customer_segment,
        COUNT(*) AS complaints_cnt
    FROM complaints_5 cpl
    LEFT JOIN transactions_4 t
      ON cpl.transaction_id = t.transaction_id
    LEFT JOIN customers_1 c
      ON cpl.customer_id = c.customer_id
    GROUP BY 1, 2, 3, 4, 5, 6, 7
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    try:
        df = conn.execute(query).df()
    except Exception as e:
        raise RuntimeError(f"SQL execution error: {e}") from e

    expected_columns = [
        "dt",
        "issue",
        "payment_rail",
        "channel",
        "merchant_category",
        "customer_segment",
        "complaints_cnt",
        "customer_id",
    ]
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in the resulting DataFrame: {missing_columns}")

    output_file = out_path / "mart_complaints_by_payment_context.parquet"
    try:
        df.to_parquet(output_file, index=False)
    except Exception as e:
        raise RuntimeError(f"Parquet save error: {e}") from e

    return df, str(output_file)

conn = duckdb.connect()
conn.register("complaints", complaints_5)
conn.register("transactions", transactions_4)
conn.register("customers", customers_1)
df_complaints_by_payment, output_path = build_mart_complaints_by_payment_context(conn)
display(df_complaints_by_payment.head(3))

# build_mart_complaints_by_payment_context
# links complaints to a specific payment context (rail/channel/MCC),
# turning “many complaints” into “a specific route is broken”

,customer_id,dt,issue,payment_rail,channel,merchant_category,customer_segment,complaints_cnt
0,CUST_003828,2025-08-16,service,rtp,web,p2p_transfer,delinquent,1
1,CUST_004023,2025-08-28,service,card,api,telecom,delinquent,1
2,CUST_008652,2025-07-04,service,card,mobile_app,travel,active,1


## KPI mart_5 - `build_mart_executive_kpis_daily`

A single-line executive view of payment volume, success, recovery, complaint pressure, and risk pressure.

In [21]:
def build_mart_executive_kpis_daily(conn, out_dir="marts"):
    query = """
    WITH tx AS (
        SELECT
            customer_id,
            DATE_TRUNC('day', CAST(transaction_ts AS TIMESTAMP)) AS dt,
            COUNT(*) AS tx_cnt,
            SUM(amount_usd) AS gross_payment_volume_usd,
            SUM(CASE WHEN status = 'approved' THEN 1 ELSE 0 END) AS approved_cnt,
            SUM(CASE WHEN status = 'approved' THEN fee_usd ELSE 0 END) AS fee_revenue_usd
        FROM transactions
        GROUP BY 1, 2
    ),
    tx_day AS (
        SELECT
            dt,
            SUM(tx_cnt) AS tx_cnt_day
        FROM tx
        GROUP BY 1
    ),
    col AS (
        SELECT
            DATE_TRUNC('day', CAST(case_open_date AS TIMESTAMP)) AS dt,
            COUNT(*) AS cases_cnt,
            SUM(CASE WHEN resolved_flag = 1 THEN 1 ELSE 0 END) AS resolved_cnt
        FROM collections_cases
        GROUP BY 1
    ),
    cmp_day AS (
        SELECT
            DATE_TRUNC('day', CAST(complaint_created_date AS TIMESTAMP)) AS dt,
            COUNT(*) AS complaints_cnt
        FROM complaints
        GROUP BY 1
    ),
    ra_day AS (
        SELECT
            DATE_TRUNC('day', CAST(alert_ts AS TIMESTAMP)) AS dt,
            COUNT(*) AS alerts_cnt
        FROM risk_alerts
        GROUP BY 1
    )
    SELECT
        tx.customer_id,
        tx.dt,
        tx.gross_payment_volume_usd,
        CASE
            WHEN COALESCE(1.0 * tx.approved_cnt / NULLIF(tx.tx_cnt, 0), 0) <> 0 THEN 1
            ELSE 0
        END AS payment_success_rate,
        ROUND(1.0 * col.resolved_cnt / NULLIF(col.cases_cnt, 0), 4) AS repayment_cure_rate,
        ROUND(1000.0 * COALESCE(cmp_day.complaints_cnt, 0) / NULLIF(tx_day.tx_cnt_day, 0), 4) AS complaint_rate_per_1k_tx,
        ROUND(1000.0 * COALESCE(ra_day.alerts_cnt, 0) / NULLIF(tx_day.tx_cnt_day, 0), 4) AS risk_alert_rate_per_1k_tx,
        tx.fee_revenue_usd
    FROM tx
    LEFT JOIN tx_day
      ON tx.dt = tx_day.dt
    LEFT JOIN col
      ON tx.dt = col.dt
    LEFT JOIN cmp_day
      ON tx.dt = cmp_day.dt
    LEFT JOIN ra_day
      ON tx.dt = ra_day.dt
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    try:
        df = conn.execute(query).df()
    except Exception as e:
        raise RuntimeError(f"SQL execution error: {e}") from e

    expected_columns = [
        "dt",
        "gross_payment_volume_usd",
        "payment_success_rate",
        "repayment_cure_rate",
        "complaint_rate_per_1k_tx",
        "risk_alert_rate_per_1k_tx",
        "fee_revenue_usd",
        "customer_id",
    ]
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in the resulting DataFrame: {missing_columns}")

    output_file = out_path / "mart_executive_kpis_daily.parquet"
    try:
        df.to_parquet(output_file, index=False)
    except Exception as e:
        raise RuntimeError(f"Parquet save error: {e}") from e

    return df, str(output_file)

conn = duckdb.connect()
conn.register("risk_alerts", risk_alerts_7)
conn.register("collections_cases", collections_cases_6)
conn.register("complaints", complaints_5)
conn.register("transactions", transactions_4)
df_executive_kpis, output_path = build_mart_executive_kpis_daily(conn)
display(df_executive_kpis.head(3))

,customer_id,dt,gross_payment_volume_usd,payment_success_rate,repayment_cure_rate,complaint_rate_per_1k_tx,risk_alert_rate_per_1k_tx,fee_revenue_usd
0,CUST_011942,2025-08-07,43.92,0,0.5000,18.2482,76.6423,0.00
1,CUST_009712,2025-05-07,95.23,1,0.6842,36.7893,90.3010,1.71
2,CUST_000918,2025-09-05,91.45,1,0.5600,13.5135,91.2162,0.50


In [22]:
# KPI 4 cards: GPV, Success Rate, Complaint Rate, Repayment Cure Rate

#1-2

# Pivot table for Gross Payment Volume (GPV) by payment_success_rate (1 = yes, 0 = no)
GPV = (
    pd.pivot_table(
        df_executive_kpis,
        index='payment_success_rate',
        values='gross_payment_volume_usd',
        aggfunc='sum',
        margins=True
    )
    .rename(columns={'gross_payment_volume_usd': 'GPV'})
)
GPV['gpv_%_from_all'] = (
    GPV['GPV'] / GPV.loc['All', 'GPV'] * 100
).round(2)
GPV
# GPV — total monetary flow through the system <- All
# Success_rate — share of successfully completed payments <- 1 means successful

,GPV,gpv_%_from_all
payment_success_rate,,
0,1433214.99,14.42
1,8506894.02,85.58
All,9940109.01,100.00


In [24]:
#3

# Pivot table for complaint_rate (per 1,000 transactions) by dates, mean
CMP_rate = pd.pivot_table(
    df_executive_kpis,
    index=['dt', 'customer_id'],
    values=['complaint_rate_per_1k_tx'],
    aggfunc='mean',
    margins=True
).sort_values(by='complaint_rate_per_1k_tx', ascending=False).head(3)
CMP_rate

# Data recalculated at the daily level: number of complaints per 1,000 transactions.
# Complaint_Rate — customer pain pressure on the flow of operations

complaint_rate_per_1k_tx
dt                  customer_id                          
2025-04-12 00:00:00 CUST_000263                      62.5
                    CUST_008193                      62.5
                    CUST_007643                      62.5

In [25]:
#4

# Pivot table for repayment_cure_rate by dates, mean
REP_Cure_Rate = pd.pivot_table(
    df_executive_kpis,
    index=['dt', 'customer_id'],
    values=['repayment_cure_rate'],
    aggfunc='mean',
    margins=True
).sort_values(by='repayment_cure_rate', ascending=False).head(3)
REP_Cure_Rate
# Repayment_Cure_Rate — effectiveness of money recovery

repayment_cure_rate
dt                  customer_id                     
2025-06-07 00:00:00 CUST_000169                  1.0
                    CUST_002854                  1.0
                    CUST_003160                  1.0

## Collections funnel_mart_6 - `build_mart_repayment_funnel_daily`

Shows how cases move toward repayment and where cure rate drops.
Funnel: past-due cases → promise-to-pay → resolved → recovered amount

In [26]:
def build_mart_repayment_funnel_daily(conn, out_dir="marts"):
    query = """
    SELECT
        c.customer_id,
        DATE_TRUNC('day', CAST(cc.case_open_date AS TIMESTAMP)) AS dt,
        cc.dpd_bucket,
        c.customer_segment,
        a.account_type,
        COUNT(*) AS opened_cases,
        SUM(CASE WHEN cc.promise_to_pay_flag = 1 THEN 1 ELSE 0 END) AS promise_to_pay_cnt,
        SUM(CASE WHEN cc.resolved_flag = 1 THEN 1 ELSE 0 END) AS resolved_cnt,
        SUM(CASE WHEN cc.resolved_flag = 1 THEN cc.outstanding_amount_usd ELSE 0 END) AS recovered_amount_usd,
        ROUND(
            1.0 * SUM(CASE WHEN cc.resolved_flag = 1 THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0),
            4
        ) AS cure_rate
    FROM collections_cases_6 cc
    LEFT JOIN customers_1 c
      ON cc.customer_id = c.customer_id
    LEFT JOIN accounts_2 a
      ON cc.account_id = a.account_id
    GROUP BY 1, 2, 3, 4, 5
    """
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    try:
        df = conn.execute(query).df()
    except Exception as e:
        raise RuntimeError(f"SQL execution error: {e}") from e

    expected_columns = [
        "dt",
        "dpd_bucket",
        "customer_segment",
        "account_type",
        "opened_cases",
        "promise_to_pay_cnt",
        "resolved_cnt",
        "recovered_amount_usd",
        "cure_rate",
        "customer_id",
    ]
    missing_columns = [col for col in expected_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in the resulting DataFrame: {missing_columns}")

    output_file = out_path / "build_mart_repayment_funnel_daily.parquet"
    try:
        df.to_parquet(output_file, index=False)
    except Exception as e:
        raise RuntimeError(f"Parquet save error: {e}") from e

    return df, str(output_file)

conn = duckdb.connect()
conn.register("collections_cases", collections_cases_6)
conn.register("accounts", accounts_2)
conn.register("customers", customers_1)
df_repayment_funnel, output_path = build_mart_repayment_funnel_daily(conn)
display(df_repayment_funnel.head(3))

,customer_id,dt,dpd_bucket,customer_segment,account_type,opened_cases,promise_to_pay_cnt,resolved_cnt,recovered_amount_usd,cure_rate
0,CUST_000005,2025-06-11,61-90,high_value,checking,1,0.0,0.0,0.00,0.0
1,CUST_000007,2025-11-14,1-7,delinquent,credit_card,1,0.0,0.0,0.00,0.0
2,CUST_000017,2025-09-22,8-30,active,credit_card,1,0.0,1.0,131.83,1.0


In [28]:
#8

# Pivot table for Recovered_Amount by sums and dates
Recovered_Amount = pd.pivot_table(
    df_repayment_funnel,
    index=['dt', 'customer_id'],
    values=['recovered_amount_usd'],
    aggfunc='sum',
    margins=True
).sort_values(by='recovered_amount_usd', ascending=False).head(3)
Recovered_Amount
# Recovered_Amount — how much money the business managed to recover:
# problem cases, overdue balances, and related recovery flows

,,recovered_amount_usd
dt,customer_id,
All,,648037.72
2025-05-03 00:00:00,CUST_001526,2576.41
2025-11-20 00:00:00,CUST_005740,2085.51


## Summary

**The value of this dashboard is not in the charts themselves, but in the sequence they create: cash-in, failure, customer pain, and recovery.**  
It shows how payment issues turn into revenue leakage, how leakage turns into complaints, and how recovery can offset part of the loss.  
For leadership, that means one screen, one logic, and faster decisions on where to act first.